# 01 - PDF Extraction + Quality Assessment
Ekstraksi teks dari PDF publikasi BPS menggunakan PyMuPDFLLM .

In [1]:
!pip install pymupdf4llm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 994.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 43.4 MB/s eta 0:00:00


In [2]:
import pprint
import importlib.metadata
import json
import logging
import pymupdf
import pymupdf4llm
from pymupdf4llm.helpers.pymupdf_rag import TocHeaders, IdentifyHeaders
import pandas as pd
from pathlib import Path

In [3]:
pymupdf4llm.use_layout(False)

## 1. Konfigurasi

In [4]:
INPUT_DIR  = Path("/kaggle/input/datasets/muhammadnabilul/bps-publications")
df_meta    = pd.read_excel(INPUT_DIR / 'List_publikasi.xlsx')
PDF_DIR    = INPUT_DIR / "pdf"
OUT_DIR    = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## Load Metadata

In [5]:
TARGET_YEARS = ["2025"]

df_meta.columns = ['tahun', 'judul', 'bps_url', 'nama_file']
df_meta['tahun'] = df_meta['tahun'].astype(str)

df_meta = (
    df_meta[
        df_meta["tahun"].isin(TARGET_YEARS)
    ]
    .reset_index(drop=True)
)

metadata_list = df_meta.to_dict(orient='records')

print(f'Total dokumen: {len(df_meta)}')
for i, row in df_meta.iterrows():
    print(f'  [{i+1:>3}] ({row["tahun"]}) {row["judul"]}')

Total dokumen: 16
  [  1] (2025) Statistik Indonesia 2025
  [  2] (2025) Analisis Petty Corruption Hasil SPAK 2020–2024
  [  3] (2025) Indeks Pembangunan Manusia 2024
  [  4] (2025) Indikator Kesejahteraan Rakyat 2025
  [  5] (2025) Indikator SDGs Kesejahteraan Rakyat 2025
  [  6] (2025) Infografis Statistik Kesejahteraan Rakyat 2025
  [  7] (2025) Profil Anak Usia Dini 2025
  [  8] (2025) Profil Statistik Kesehatan 2025
  [  9] (2025) Statistik Kesejahteraan Rakyat 2025
  [ 10] (2025) Statistik Kriminal 2024/2025
  [ 11] (2025) Statistik Pemuda Indonesia 2025
  [ 12] (2025) Statistik Pendidikan 2025
  [ 13] (2025) Statistik Penduduk Lanjut Usia 2025
  [ 14] (2025) Statistik Penunjang Pendidikan 2024
  [ 15] (2025) Statistik Politik 2025
  [ 16] (2025) Statistik Sosial Budaya 2024


## 2. Bahan Ekstraksi

In [6]:
EXTRACTION_CONFIG = {
    "page_chunks"    : True,
    "extract_words"  : True,
    "force_text"     : True,
    "ignore_code"    : False,
    "ignore_images"  : False,
    "ignore_graphics": False,
    "ignore_alpha"   : False,
    "detect_bg_color": True,
    "table_strategy" : "lines_strict",
    "image_size_limit": 0.01,
    "fontsize_limit"  : 3,
    "graphics_limit"  : None,
    "embed_images"    : False,
    "write_images"    : False,
    "show_progress"   : True,
    "margins"         : 0,
}

## Fungsi Ekstraksi
Selengkapnya bisa dilihat di https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/api.html

In [7]:
def extract_pdf(pdf_path: Path, row: pd.Series) -> tuple[list, dict]:
    doc  = pymupdf.open(pdf_path)
    toc  = doc.get_toc()
    hdr_info = TocHeaders(doc) if toc else IdentifyHeaders(doc)
 
    chunks = pymupdf4llm.to_markdown(
        doc,
        hdr_info=hdr_info,
        **EXTRACTION_CONFIG,
    )
 
    doc_metadata = doc.metadata
    doc.close()
 
    pages         = []
    flagged_pages = []
 
    for chunk in chunks:
        text    = chunk.get("text", "")
        n_chars = len(text)
        n_words = len(chunk.get("words", []))
        boxes   = chunk.get("page_boxes", [])
        box_classes = list({b.get("class", "unknown") for b in boxes})
 
        flags = []
        if n_chars == 0:
            flags.append("empty")
        if 0 < n_chars < 100:
            flags.append("low_char")
        if "page-header" in box_classes or "page-footer" in box_classes:
            flags.append("has_header_footer")
        if len(chunk.get("images", [])) > 0 and n_chars < 200:
            flags.append("image_dominant")
 
        page_entry = {
            "page_number": chunk["metadata"]["page"],
            "file_path"  : chunk["metadata"]["file_path"],
            "chunk_id"   : f"{pdf_path.stem}__p{chunk['metadata']['page']:04d}",
            "text_md"    : text,
            "toc_items"  : chunk.get("toc_items",  []),
            "tables"     : chunk.get("tables",     []),
            # "images"     : chunk.get("images",     []),
            "graphics"   : chunk.get("graphics",   []),
            "words"      : chunk.get("words",      []),
            "flags"      : flags,
            "raw_metrics": {
                "char_count"     : n_chars,
                "token_estimate" : n_chars // 4,
                "line_count"     : len(text.splitlines()),
                "word_count"     : n_words,
                "table_count"    : len(chunk.get("tables",   [])),
                "image_count"    : len(chunk.get("images",   [])),
                "graphic_count"  : len(chunk.get("graphics", [])),
            },
        }
 
        pages.append(page_entry)
        if flags:
            flagged_pages.append({
                "page_number": chunk["metadata"]["page"],
                "flags"      : flags,
            })
 
    total_halaman  = len(pages)
    total_chars    = sum(p["raw_metrics"]["char_count"]   for p in pages)
    total_words    = sum(p["raw_metrics"]["word_count"]   for p in pages)
    total_tables   = sum(p["raw_metrics"]["table_count"]  for p in pages)
    total_images   = sum(p["raw_metrics"]["image_count"]  for p in pages)
    n_empty        = sum(1 for p in pages if "empty" in p["flags"])
    coverage_ratio = (total_halaman - n_empty) / total_halaman if total_halaman else 0
 
    if coverage_ratio >= 0.70:
        overall_quality = "good"
    elif coverage_ratio >= 0.50:
        overall_quality = "review"
    else:
        overall_quality = "poor"
 
    quality = {
        "total_halaman"           : total_halaman,
        "halaman_diekstrak"       : total_halaman - n_empty,
        "halaman_preliminary_skip": 0,
        "halaman_empty"           : n_empty,
        "coverage_ratio"          : round(coverage_ratio, 3),
        "flagged_pages"           : flagged_pages,
        "n_flagged"               : len(flagged_pages),
        "overall_quality"         : overall_quality,
        "doc_stats": {
            "total_chars" : total_chars,
            "total_words" : total_words,
            "total_tables": total_tables,
            "total_images": total_images,
        },
        "doc_metadata": doc_metadata,
        "doc_toc"     : toc,
        "extraction_info": {
            "pymupdf_version"    : importlib.metadata.version("pymupdf"),
            "pymupdf4llm_version": importlib.metadata.version("pymupdf4llm"),
            "use_layout"         : False,
            "hdr_info"           : "TocHeaders" if toc else "IdentifyHeaders",
            "ocr"                : False,
            **{k: v for k, v in EXTRACTION_CONFIG.items()
               if k in ("table_strategy", "extract_words",
                        "detect_bg_color", "fontsize_limit")},
        },
    }
 
    return pages, quality

In [8]:
def sanitize(obj):

    if isinstance(obj, pymupdf.Rect):
        return list(obj)

    if isinstance(obj, dict):
        return {
            k: sanitize(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            sanitize(v)
            for v in obj
        ]

    return obj

## 3. Main Program

In [9]:
all_results     = []
quality_reports = []
total           = len(df_meta)
 
for idx, row in df_meta.iterrows():
    tahun     = str(row['tahun'])
    nama_file = row['nama_file']
    judul     = row['judul']
    pdf_path  = PDF_DIR / tahun / nama_file
 
    print(f'[{idx+1}/{total}] {judul[:60]}')
 
    if not pdf_path.exists():
        print(f'  ✗ File tidak ditemukan: {pdf_path}')
        quality_reports.append({
            'nama_file': nama_file, 'judul': judul, 'tahun': tahun,
            'recommendation': 'file-missing', 'overall_quality': 'poor',
        })
        continue
 
    try:
        pages, quality = extract_pdf(pdf_path, row)
    except Exception as e:
        print(f'  ✗ Error: {e}')
        quality_reports.append({
            'nama_file': nama_file, 'judul': judul, 'tahun': tahun,
            'recommendation': 'error', 'overall_quality': 'poor', 'error': str(e),
        })
        continue
 
    quality.update({'nama_file': nama_file, 'judul': judul, 'tahun': tahun})
    quality_reports.append(quality)
 
    doc_result = {
        'judul'    : judul,
        'tahun'    : tahun,
        'nama_file': nama_file,
        'bps_url'  : row.get('bps_url', ''),
        'n_halaman': quality['total_halaman'],
        'n_chunks' : len(pages),
        'quality'  : quality['overall_quality'],
        'pages'    : pages,
    }
    doc_result = sanitize(doc_result)
    
    all_results.append(doc_result)
    
    fname = nama_file.replace('.pdf', '.json')
    
    with open(OUT_DIR / fname, 'w', encoding='utf-8') as f:
        json.dump(doc_result, f, ensure_ascii=False, indent=2)
 
    with open(OUT_DIR / 'quality_report.json', 'w', encoding='utf-8') as f:
        json.dump(quality_reports, f, ensure_ascii=False, indent=2)
 
    print(f'  ✓ {quality["halaman_diekstrak"]}/{quality["total_halaman"]} hal'
          f' | skip_prelim={quality["halaman_preliminary_skip"]}'
          f' | {quality["overall_quality"]}')
 
    if quality['flagged_pages']:
        print(f'  ⚑ {quality["n_flagged"]} halaman di-flag')
 
print(f'\nSelesai: {len(all_results)} dokumen')

[1/16] Statistik Indonesia 2025
Processing /kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/statistik-indonesia-2025.pdf...


100%|██████████| 898/898 [05:15<00:00,  2.85it/s]


  ✓ 864/898 hal | skip_prelim=0 | good
  ⚑ 157 halaman di-flag
[2/16] Analisis Petty Corruption Hasil SPAK 2020–2024
Processing /kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/analisis-petty-corruption-hasil-spak-2020---2024.pdf...


100%|██████████| 194/194 [00:43<00:00,  4.46it/s]


  ✓ 173/194 hal | skip_prelim=0 | good
  ⚑ 28 halaman di-flag
[3/16] Indeks Pembangunan Manusia 2024
Processing /kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/indeks-pembangunan-manusia-2024.pdf...


...

  ✓ 180/200 hal | skip_prelim=0 | good
  ⚑ 73 halaman di-flag
[15/16] Statistik Politik 2025
Processing /kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/statistik-politik-2025.pdf...


100%|██████████| 208/208 [01:21<00:00,  2.55it/s]


  ✓ 186/208 hal | skip_prelim=0 | good
  ⚑ 25 halaman di-flag
[16/16] Statistik Sosial Budaya 2024
Processing /kaggle/input/datasets/muhammadnabilul/bps-publications/pdf/2025/statistik-sosial-budaya-2024.pdf...


100%|██████████| 228/228 [00:47<00:00,  4.77it/s]


  ✓ 214/228 hal | skip_prelim=0 | good
  ⚑ 64 halaman di-flag

Selesai: 16 dokumen
